# Electric-Machine Cogging And Skew Demo

This result-saved notebook promotes the remaining electric-machine demo out of `examples/`. The executable helper now lives beside this notebook as `cogging_skew_demo.py`; validation-class scripts live under `validation_test/electric_machine/`.

## Method

A diametric permanent-magnet rotor inside a two-fold salient stator produces a reluctance/cogging torque dominated by order 2. The helper solves the planar magnetostatic problem over a rotor-angle sweep, extracts torque with the weighted Maxwell-stress eggshell integral, then converts the per-depth 2-D result into a physical torque using `MachineScaling`.

For continuous skew, each harmonic is reduced by the analytic sinc-like `skew_factor(n, skew)`. The notebook checks the computed skew-averaged curve against that factor on the actual finite-element torque curve.

In [1]:
from pathlib import Path
import json
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'cogging_skew_demo.py').exists():
    NOTEBOOK_DIR = Path('docs/electric_machine').resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

from cogging_skew_demo import run_demo, write_results_json

print(f"notebook_dir={NOTEBOOK_DIR}")

notebook_dir=\\192.168.11.100\work\00_CAE\Radia\01_GitHub\docs\electric_machine


In [2]:
result = run_demo(verbose=True)
json_path = write_results_json(result, NOTEBOOK_DIR / 'cogging_skew_demo_results.json')
print(f'\nwrote {json_path.relative_to(NOTEBOOK_DIR)}')

1) tau(theta): dominant order = 2 (expect 2), mean/amp = 0.2% (expect ~0)
2) MachineScaling(50 mm stack): peak |tau| = 526.7894 mN*m (= 2D x length_unit^2 x stack)
3) skew_average(tau) vs analytic skew_factor (order-2 ripple):
   skew    30 deg : amp  0.955 x  (skew_factor predicts  0.955)
   skew    90 deg : amp  0.637 x  (skew_factor predicts  0.637)
   skew   180 deg : amp  0.000 x  (skew_factor predicts  0.000)  <- NULLS the order-2 ripple

[OK] electric-machine demo: tau(theta) reluctance ripple (order 2, zero mean), MachineScaling -> physical N*m, and skew_average == analytic skew_factor on the real FE curve (a half-period skew nulls the ripple). Pure radia-ngsolve, analytic-gated.

wrote cogging_skew_demo_results.json


In [3]:
data = json.loads((NOTEBOOK_DIR / 'cogging_skew_demo_results.json').read_text(encoding='utf-8'))
assert data['schema'] == 'radia.docs.electric_machine.cogging_skew_demo.v1'
assert all(data['checks'].values()), data['checks']
assert data['dominant_order'] == 2
assert data['mean_fraction'] < 0.15
assert abs(data['skew_rows'][-1]['amplitude_ratio']) < 1e-12

print(f"dominant_order={data['dominant_order']}")
print(f"mean_fraction={data['mean_fraction']:.6f}")
print(f"peak_torque_mNm={data['peak_torque_mNm']:.4f}")
print('checks=' + json.dumps(data['checks'], sort_keys=True))

dominant_order=2
mean_fraction=0.002156
peak_torque_mNm=526.7894
checks={"dominant_order_is_2": true, "mean_fraction_lt_0p15": true, "skew_matches_analytic": true}


In [4]:
from IPython.display import Markdown, display

rows = ['| skew [deg] | FE amplitude ratio | analytic factor |', '|---:|---:|---:|']
for row in data['skew_rows']:
    rows.append(f"| {row['skew_deg']:.0f} | {row['amplitude_ratio']:.6f} | {row['predicted_ratio']:.6f} |")
display(Markdown('\n'.join(rows)))

| skew [deg] | FE amplitude ratio | analytic factor |
|---:|---:|---:|
| 30 | 0.954930 | 0.954930 |
| 90 | 0.636620 | 0.636620 |
| 180 | 0.000000 | 0.000000 |

## Durable Artifacts

- `cogging_skew_demo.py` is the notebook-coupled helper.
- `cogging_skew_demo_results.json` stores the computed values, checks, runtime versions, and generation timestamp.
- `cogging_skew_demo_result.json` stores the saved-output notebook checksum for docs-policy synchronization.